# 04. Model Training

In [1]:
# 01. APRENDIZADO DE MÁQUINA (CLASSIFICAÇÃO PREDITIVA)

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings

warnings.filterwarnings("ignore")

print("="*80)
print(" Iniciando avaliação do modelo final (XGBOOST + K=15)")
print("="*80)

# 1. Carregamento e Isolamento da Tarefa
caminho_csv = '../reports/tabela_features_eeg_completa.csv'
df = pd.read_csv(caminho_csv)

SEED = 97
cv_strategy = LeaveOneOut() # LOOCV (N=42)

condicoes_busca = {
    'Face Feliz': ['Face Feliz'],
    'Face Neutra': ['Face Neutra'],
    'Face Raiva': ['Face Raiva']
}

# 2. Pipeline Final
pipeline_classificacao = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=15)),
    ('clf', XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, 
                          eval_metric='logloss', random_state=SEED))
])

# 3. Validação e Extração de Métricas
for nome_condicao, lista_triggers in condicoes_busca.items():
    print(f"\n" + "-"*60)
    print(f"ESTÍMULO ANALISADO: {nome_condicao.upper()}")
    print("-" * 60)
    
    df_f = df[df['Condicao'].isin(lista_triggers)].copy()
    
    if df_f.empty:
        print(f"Atenção: Sem dados para {nome_condicao}.")
        continue

    # Garantia de integridade amostral (N=42)
    df_f = df_f.groupby(['ID', 'Grupo']).mean(numeric_only=True).reset_index()
    print(f"Amostra consolidada: {df_f.shape[0]} sujeitos.\n")

    y = df_f['Grupo'].apply(lambda x: 1 if 'TEA' in x else 0).values
    X = df_f.drop(columns=['ID', 'Grupo', 'Condicao', 'Tipo'], errors='ignore')

    # Predição Cega (LOOCV)
    y_pred = cross_val_predict(pipeline_classificacao, X, y, cv=cv_strategy)

    acc = accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    print(f" ACURÁCIA GLOBAL DO SISTEMA: {acc:.2%}\n")

    print("Matriz de Confusão:")
    print(f"                   Predito Controle (0) | Predito TEA (1)")
    print(f"Real Controle (0) |        {cm[0,0]:02d}           |        {cm[0,1]:02d}")
    print(f"Real TEA (1)      |        {cm[1,0]:02d}           |        {cm[1,1]:02d}\n")

    print("Relatório de Desempenho (Precision, Recall, F1-Score):")
    report = classification_report(y, y_pred, target_names=['Controle', 'TEA'], digits=3)
    print(report)

print("="*80)

 Iniciando avaliação do modelo final (XGBOOST + K=15)

------------------------------------------------------------
ESTÍMULO ANALISADO: FACE FELIZ
------------------------------------------------------------
Amostra consolidada: 42 sujeitos.

 ACURÁCIA GLOBAL DO SISTEMA: 83.33%

Matriz de Confusão:
                   Predito Controle (0) | Predito TEA (1)
Real Controle (0) |        21           |        03
Real TEA (1)      |        04           |        14

Relatório de Desempenho (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

    Controle      0.840     0.875     0.857        24
         TEA      0.824     0.778     0.800        18

    accuracy                          0.833        42
   macro avg      0.832     0.826     0.829        42
weighted avg      0.833     0.833     0.833        42


------------------------------------------------------------
ESTÍMULO ANALISADO: FACE NEUTRA
------------------------------------------------------------


In [3]:
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings("ignore")

print(" Calculando métricas clínicas globais para todas as Faces...\n")

# Carregamento dos Dados
caminho_csv = '../reports/tabela_features_eeg_completa.csv'
try:
    df = pd.read_csv(caminho_csv)
except FileNotFoundError:
    df = pd.read_csv('reports/tabela_features_eeg_completa.csv')

condicoes = {
    'FF': 'Face Feliz',
    'FN': 'Face Neutra',
    'FR': 'Face Raiva'
}

# Cabeçalho da Tabela
print("Tabela - Acurácia Global do modelo XGBoost para os dados de EEG")
print("-" * 75)
print(f"{'EEG':<5} | {'Acurácia':<10} | {'Sensibilidade':<15} | {'Precisão':<10} | {'F1-Score':<10} | {'AUC-ROC':<10}")
print("-" * 75)

for sigla, condicao in condicoes.items():
    df_cond = df[df['Condicao'] == condicao].copy()
    
    if df_cond.empty:
        print(f"{sigla:<5} | Sem dados para esta condição")
        continue

    # Agrupamento e definição de variáveis
    df_cond = df_cond.groupby(['ID', 'Grupo']).mean(numeric_only=True).reset_index()
    y = df_cond['Grupo'].apply(lambda x: 1 if 'TEA' in x else 0).values
    X = df_cond.drop(columns=['ID', 'Grupo', 'Condicao', 'Tipo'], errors='ignore')

    # Pipeline
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', RobustScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=15)),
        ('clf', XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, 
                              eval_metric='logloss', random_state=97))
    ])

    # Predições LOOCV
    cv = LeaveOneOut()
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)
    y_probs = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

    # Cálculos Matemáticos
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y, y_pred) * 100
    sens = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0
    prec = (precision_score(y, y_pred, zero_division=0) * 100)
    f1 = (f1_score(y, y_pred, zero_division=0) * 100)
    
    # Transformando AUC para formato de porcentagem
    auc = (roc_auc_score(y, y_probs) * 100) 

    # Impressão da linha formatada
    print(f"{sigla:<5} | {acc:>6.2f}%   | {sens:>12.2f}%  | {prec:>7.2f}%  | {f1:>7.2f}%  | {auc:>7.2f}%")

print("-" * 75)

 Calculando métricas clínicas globais para todas as Faces...

Tabela - Acurácia Global do modelo XGBoost para os dados de EEG
---------------------------------------------------------------------------
EEG   | Acurácia   | Sensibilidade   | Precisão   | F1-Score   | AUC-ROC   
---------------------------------------------------------------------------
FF    |  83.33%   |        77.78%  |   82.35%  |   80.00%  |   74.07%
FN    |  57.14%   |        50.00%  |   50.00%  |   50.00%  |   63.66%
FR    |  64.29%   |        44.44%  |   61.54%  |   51.61%  |   68.75%
---------------------------------------------------------------------------
